# Gemma4 test

In [ ]:
#@title Librerías necesarias
import json
import random
import torch
!pip install unsloth codecarbon
import unsloth
from unsloth import FastVisionModel
from codecarbon import EmissionsTracker
import gc
import re
import os
from google.colab import drive
from PIL import Image
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#@title Montar Google Drive
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

QUESTIONS_FILE = os.path.join(BASE_PATH, "data/test/multiple_choice_dataset.json")

Mounted at /content/drive


In [ ]:
SYSTEM_PROMPT_TEXT = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer atentamente el texto y responder ÚNICAMENTE con la letra de la opción correcta.
Las ÚNICAS opciones de respuesta válidas son: {letras_validas}

Responde a la pregunta basándote EXCLUSIVAMENTE en la información del texto, sin utilizar conocimiento externo.

No escribas explicaciones, ni introducciones, ni repitas la pregunta.
SOLO la letra de la opción elegida."""


In [ ]:
def load_model_unsloth(model_name, max_seq_length=2048, dtype=None, load_in_4bit=True):
    """
    Carga un modelo y su tokenizador usando Unsloth y lo prepara para inferencia.
    """
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    FastVisionModel.for_inference(model)

    return model, tokenizer

In [ ]:
def load_data():
    """Carga los ficheros JSON para evaluar los modelos."""
    with open(QUESTIONS_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

In [ ]:
def filter_questions(data):
  """Filtra las preguntas del subset y prepara la lista de tareas a procesar."""
  tareas = []
  for exam in data['exams']:
      nivel = exam['level']
      for ex_wrapper in exam['exercises']:
          exercise = ex_wrapper['exercise']

          for q in exercise['questions']:
              q_id = q['questionId']
              tareas.append({
                  "id": q_id,
                  "nivel": nivel,
                  "contexto": exercise.get('text', ''),
                  "pregunta": q['text'],
                  "opciones": q['options']
              })
  return tareas

In [ ]:
def prepare_batch(batch, text_template, output_mode):
    """
    Construye los mensajes inyectando el System Prompt adecuado
    (texto o multimodal) dependiendo de si la tarea incluye imágenes.
    """
    mensajes_batch = []
    imagenes_batch = []

    for t in batch:
        user_content = []
        imagenes_tarea = []

        options = [opt['optionId'] for opt in t['opciones']]
        valid_options = ", ".join(options)

        # Apply system prompt template
        system_prompt = text_template.format(letras_validas=valid_options)

        base_text = f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n"
        user_content.append({"type": "text", "text": base_text})

        for opt in t['opciones']:
            letra = opt['optionId']
            texto = opt.get('text', '').strip()
            ruta_img = opt.get('image-path', '')

            if texto:
                user_content.append({"type": "text", "text": f"{letra}) {texto}\n"})

            elif ruta_img:
                user_content.append({"type": "text", "text": f"{letra}) "})


                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)


                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n"})

        if output_mode == "letra":
            user_content.append({"type": "text", "text": "\nRespuesta:"})

        mensajes_batch.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ])
        imagenes_batch.append(imagenes_tarea)

    return mensajes_batch, imagenes_batch

In [ ]:
def generate_response(model, tokenizer, batch_messages, batch_imagenes, max_new_tokens):
    """Ejecuta la inferencia multimodal sobre un lote y devuelve los textos generados."""

    textos_prompt = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    imagenes_planas = [img for sublista in batch_imagenes for img in sublista]

    model_inputs = tokenizer(
        text=textos_prompt,
        images=imagenes_planas if len(imagenes_planas) > 0 else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    respuestas_brutas = []

    for output in outputs:
        gen_tokens = output[input_len:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

In [ ]:
def process_response(texto_bruto, modo_salida):
    """Extrae la letra (A-D) y la explicación según el formato esperado."""
    prediccion = "N/A"
    explicacion = ""
    error_formato = False

    if modo_salida == "json":
        explicacion = texto_bruto
        try:
            json_match = re.search(r'\{.*\}', texto_bruto, re.DOTALL)
            if json_match:
                datos = json.loads(json_match.group(0))
                letra_raw = datos.get("respuesta", "").strip().upper()
                match_letra = re.search(r'[A-D]', letra_raw)
                prediccion = match_letra.group(0) if match_letra else "N/A"
                explicacion = datos.get("razonamiento", "")
            else:
                error_formato = True
        except Exception:
            error_formato = True

    elif modo_salida == "letra":
        texto_bruto = texto_bruto.upper()
        match = re.search(r'[A-D]', texto_bruto)
        prediccion = match.group(0) if match else "N/A"
        if not match:
            error_formato = True
    return prediccion, explicacion, error_formato

In [ ]:
def split_tasks_by_modality(tareas):
    """Separa las tareas en dos grupos: de solo texto y con imágenes."""
    tareas_texto = []
    tareas_imagen = []

    for t in tareas:
        tiene_imagen = False
        if isinstance(t.get('opciones'), list):
            tiene_imagen = any(opt.get('image-path', '') != '' for opt in t['opciones'])

        if tiene_imagen:
            tareas_imagen.append(t)
        else:
            tareas_texto.append(t)

    return tareas_texto, tareas_imagen

In [ ]:
def run_inference(
    model,
    tokenizer,
    system_prompt_text,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size_texto=4,
    output_file="resultados.jsonl"
):
    """Ejecuta la inferencia procesando primero texto en batches y luego imágenes 1 a 1."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data = load_data()
    todas_las_tareas = filter_questions(data)
    tareas_texto, tareas_imagen = split_tasks_by_modality(todas_las_tareas)

    if os.path.exists(output_file):
        os.remove(output_file)

    def procesar_grupo(grupo_tareas, b_size, descripcion):
        for i in tqdm(range(0, len(grupo_tareas), b_size), desc=descripcion):
            batch = grupo_tareas[i : i + b_size]

            mensajes, imagenes = prepare_batch(batch, system_prompt_text, modo_salida)
            textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

            batch_results = []
            for j, texto_bruto in enumerate(textos_generados):
                prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

                tarea_actual = batch[j]
                nivel = tarea_actual["nivel"]

                batch_results.append({
                    "questionId": tarea_actual["id"],
                    "nivel": nivel,
                    "pregunta": tarea_actual["pregunta"],
                    "prediccion_modelo": prediccion,
                    "respuesta_completa": texto_bruto,
                    "explicacion": explicacion,
                    "error_procesamiento_json": errors,
                })

            with open(output_file, 'a', encoding='utf-8') as f:
                for resultado in batch_results:
                    linea_json = json.dumps(resultado, ensure_ascii=False)
                    f.write(linea_json + '\n')

    if tareas_texto:
        print(f"\n--- Procesando {len(tareas_texto)} tareas de SOLO TEXTO (Batch Size: {batch_size_texto}) ---")
        procesar_grupo(tareas_texto, batch_size_texto, "Progreso Texto")

    if tareas_imagen:
        print(f"\n--- Procesando {len(tareas_imagen)} tareas MULTIMODALES (Batch Size: 1) ---")
        procesar_grupo(tareas_imagen, 1, "Progreso Imágenes")

    print(f"\nResultados guardados en: {output_file}")

## [Gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it) cuantizado


In [ ]:
MODEL_PATH = os.path.join(BASE_PATH, "ft_gemma4")
gemma4_model, gemma4_tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit = True,
)

FastVisionModel.for_inference(gemma4_model)

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (language_model): Gemma4TextModel(
          (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
          (layers): ModuleList(
            (0): Gemma4TextDecoderLayer(
              (self_attn): Gemma4TextAttention(
                (q_norm): Gemma4RMSNorm()
                (k_norm): Gemma4RMSNorm()
                (v_norm): Gemma4RMSNorm()
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=2560, out_features=512, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=2560, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=16, out_features=512, bias

In [ ]:
gemma4_path = os.path.join(BASE_PATH, 'gemma4_results', 'test')
gemma4_results_path = os.path.join(gemma4_path, "ft_gemma4_test.json")
test_gemma4_processed_path = os.path.join(gemma4_path, "ft_gemma4_test_formatted.json")

### zero-shot

In [ ]:
gemma4_tokenizer.padding_side = "left"
if gemma4_tokenizer.pad_token is None:
    gemma4_tokenizer.pad_token = gemma4_tokenizer.eos_token

# Configure emissions tracker
tracker = EmissionsTracker(
    project_name="ft_gemma4_test",
    output_dir=gemma4_path,
    output_file="ft_gemma4_test_emissions.csv",
    log_level="warning"
)

tracker.start()
try:
  run_inference(
      model=gemma4_model,
      tokenizer=gemma4_tokenizer,
      system_prompt_text=SYSTEM_PROMPT_TEXT,
      modo_salida="letra",
      max_new_tokens=5,
      batch_size_texto=16,
      output_file=gemma4_results_path
  )
finally:
  emisiones = tracker.stop()
  print(f"Inferencia completada.")
  print(f"Emisiones estimadas: {emisiones:.4f} kg de CO2eq")

[codecarbon WARNING @ 16:06:13] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 16:06:13] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 16:06:13] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon WARNING @ 16:06:14] Unable to access geographical location through primary API. Will resort to using the backup API - Exception : Region is empty - url=https://get.geojs.io/v1/ip/geo.json



--- Procesando 1784 tareas de SOLO TEXTO (Batch Size: 16) ---


Progreso Texto: 100%|██████████| 112/112 [02:30<00:00,  1.34s/it]



--- Procesando 46 tareas MULTIMODALES (Batch Size: 1) ---


Progreso Imágenes: 100%|██████████| 46/46 [03:27<00:00,  4.52s/it]


Resultados guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/test/ft_gemma4_test.json
Inferencia completada.
Emisiones estimadas: 0.0104 kg de CO2eq


In [ ]:
def procesar_json(archivo_entrada, archivo_salida):
    resultado = {}

    # Abrimos el archivo de entrada en modo lectura
    with open(archivo_entrada, 'r', encoding='utf-8') as f_entrada:
        for linea in f_entrada:
            # Limpiamos los espacios en blanco y saltos de línea
            linea = linea.strip()
            if linea: # Verificamos que la línea no esté vacía
                try:
                    # Convertimos la cadena de texto a un diccionario de Python
                    datos = json.loads(linea)

                    # Extraemos los valores deseados
                    q_id = datos.get("questionId")
                    prediccion = datos.get("prediccion_modelo")

                    # Si ambas claves existen, las agregamos al diccionario final
                    if q_id and prediccion is not None:
                        resultado[q_id] = prediccion

                except json.JSONDecodeError:
                    print(f"Error al procesar la línea: {linea}")

    # Guardamos el diccionario resultante en un nuevo archivo JSON
    with open(archivo_salida, 'w', encoding='utf-8') as f_salida:
        # indent=4 le da el formato estructurado y legible que buscas
        json.dump(resultado, f_salida, indent=4, ensure_ascii=False)

procesar_json(gemma4_results_path, test_gemma4_processed_path)
print(f"Procesamiento completado. El archivo ha sido guardado como '{test_gemma4_processed_path}'.")

Procesamiento completado. El archivo ha sido guardado como '/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/test/ft_gemma4_test_formatted.json'.
